# Chapter 42 — Single View Metrology

Companion tutorial for **[Foundations of Computer Vision](https://visionbook.mit.edu/3d_scene_understanding_single_view.html)** by Torralba, Isola, and Freeman (MIT Press, 2024).

The question: given one photograph, can you measure heights, recover camera intrinsics, and locate 3D points? The answer is yes — provided you can find vanishing points and at least one known length in the scene. The chapter is a tour of the geometric tools (cross-ratio, horizon line, orthogonality constraints on the camera intrinsics) that make this work.

## How this notebook handles the chapter's photographs

Of the chapter's **20 figures, 9 are real photographs** of a specific office (bookshelf, desk, red bottle), an Ames room, the Leaning Tower of Pisa, and a calibration ruler. We can't reshoot those. The strategy this notebook takes:

* **Build a synthetic 3D office scene** in PyTorch (a room with a floor, two walls, a bookshelf, a desk, a bottle) and project it through a chosen camera matrix. The rendered "photograph" then serves every place the book uses its office photo (figures 42.1, 42.10, 42.13, 42.14, 42.16, 42.17, 42.19).
* **A synthetic scene is actually better than a real photo** for teaching single-view metrology — we know the ground-truth heights and camera intrinsics, so we can verify that the cross-ratio measurements and the calibration algorithm recover them correctly.
* **Perceptual demos** (Ames room 42.3–42.4; Pisa 42.7) are noted but not rebuilt — they're illusions designed for the eye, not the algorithm.

## Status

This notebook is currently a **structural scaffold** with the math, the synthetic scene generator, and the central algorithms (vanishing-point detection from edges, cross-ratio height measurement, camera calibration from vanishing points). Several figure-specific renders are marked `# TODO` and will be filled in iteratively — see the open-questions cell at the end.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

torch.set_default_dtype(torch.float32)
torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

## A synthetic office, end to end

We build a small office scene as a list of 3D line segments. Each segment is one edge of the room, the bookshelf, the desk, or the bottle. World coordinates are in centimetres. The camera looks into the room from a known position with known intrinsics — all of this is ground truth that we will *recover* later in the chapter using only the projected 2D image.

In [ ]:
def box_edges(x0, y0, z0, dx, dy, dz):
    """12 edges of an axis-aligned box, returned as (12, 2, 3) world-coord segments."""
    c = torch.tensor([
        [x0, y0, z0], [x0 + dx, y0, z0], [x0 + dx, y0 + dy, z0], [x0, y0 + dy, z0],
        [x0, y0, z0 + dz], [x0 + dx, y0, z0 + dz], [x0 + dx, y0 + dy, z0 + dz], [x0, y0 + dy, z0 + dz],
    ])
    idx = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
           (0, 4), (1, 5), (2, 6), (3, 7)]
    return torch.stack([torch.stack([c[a], c[b]], dim=0) for a, b in idx], dim=0)


def build_office():
    """Returns a dict of named edge sets; coordinates in cm."""
    room = box_edges(0, 0, 0, 250, 220, 350)              # 2.5 m wide x 2.2 m tall x 3.5 m deep
    bookshelf = box_edges(15, 0, 280, 80, 197, 35)        # height 197 cm — the chapter's reference object
    desk = box_edges(110, 0, 180, 100, 76, 60)            # height 76 cm — the target of Algorithm 2
    bottle = box_edges(150, 76, 200, 8, 25, 8)            # height 25 cm, sitting ON the desk
    return {"room": room, "bookshelf": bookshelf, "desk": desk, "bottle": bottle}


def look_at(eye, target, up=(0.0, 1.0, 0.0)):
    """World-from-camera rotation R and translation t (camera-from-world transform: P_c = R(P_w - eye))."""
    eye = torch.as_tensor(eye, dtype=torch.float32)
    target = torch.as_tensor(target, dtype=torch.float32)
    up = torch.as_tensor(up, dtype=torch.float32)
    f = (target - eye); f = f / f.norm()
    r = torch.linalg.cross(f, up); r = r / r.norm()
    u = torch.linalg.cross(r, f)
    R = torch.stack([r, u, f], dim=0)   # rows = camera basis in world coords
    return R, eye


def make_K(fx, fy, cx, cy):
    return torch.tensor([[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]])


def project_segments(segments, K, R, eye):
    """Project (N, 2, 3) world segments to (N, 2, 2) image-plane pixel coords."""
    P_c = (segments - eye) @ R.T               # camera coords
    p_h = P_c @ K.T                            # (N, 2, 3) homogeneous image
    return p_h[..., :2] / p_h[..., 2:3]

In [ ]:
# The ground-truth scene we will be "a single view of".
scene = build_office()
K_true = make_K(fx=3054.6, fy=3054.6, cx=1999.4, cy=1528.3)   # the K the book reports for its office photo
R_true, eye_true = look_at(eye=(50.0, 163.0, 30.0), target=(140.0, 100.0, 280.0))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
for name, color in [("room", "gray"), ("bookshelf", "saddlebrown"), ("desk", "steelblue"), ("bottle", "crimson")]:
    for seg in scene[name]:
        ax3d.plot(seg[:, 0], seg[:, 2], seg[:, 1], color=color, lw=1.2)
ax3d.set_xlabel("X (cm)"); ax3d.set_ylabel("Z (cm)"); ax3d.set_zlabel("Y (cm)")
ax3d.set_title("Ground-truth 3D scene")
ax3d.view_init(elev=15, azim=-72)

ax2d = fig.add_subplot(1, 2, 2)
for name, color in [("room", "gray"), ("bookshelf", "saddlebrown"), ("desk", "steelblue"), ("bottle", "crimson")]:
    proj = project_segments(scene[name], K_true, R_true, eye_true)
    for seg in proj:
        ax2d.plot(seg[:, 0], seg[:, 1], color=color, lw=1.0)
ax2d.set_xlim(0, 2 * K_true[0, 2].item()); ax2d.set_ylim(2 * K_true[1, 2].item(), 0)
ax2d.set_aspect("equal")
ax2d.set_title("Projected single view (stand-in for Fig 42.1, 42.10, 42.13...)")

plt.tight_layout()
plt.show()

## 42.3 — Linear perspective

A 3D line $\mathbf{P}(t) = \mathbf{P}_0 + t\,\mathbf{D}$ projects to a 2D line that, as $t \to \infty$, terminates at the **vanishing point**
$$\mathbf{v} = f\,\big(D_X / D_Z, \; D_Y / D_Z\big)$$
(equation 42.3). Equation 42.5 generalises this through the full camera matrix:
$$\mathbf{v} = \mathbf{K}\mathbf{R}\mathbf{D}.$$

Two consequences we'll use repeatedly:
1. The vanishing point depends only on the line's **direction**, not on where it starts. So all lines parallel in 3D meet at the same image point.
2. All vanishing points of lines lying in a single plane lie on the plane's **horizon line**.

*[Figures 42.5 (3D-line projection diagram), 42.6 (parallel-line convergence sketch), 42.8 (horizon-line diagram on building facade) — to fill in alongside Ch. 47's vanishing-point cells which already cover the core math.]*

### 42.3.4 — Detecting vanishing points

**Algorithm 1 (book § 42.3.4).** Given a set of 2D line segments believed to be parallel in 3D:

1. For each *pair* of segments, compute their intersection in the image plane (cross product of their homogeneous line vectors).
2. RANSAC over those candidate intersections: a vanishing point is a location with many votes.
3. Repeat per direction (the office has three orthogonal world directions → three vanishing points).

We implement this against the projected office scene — we know the ground-truth vanishing points (from $\mathbf{K}\mathbf{R}\mathbf{D}$) so we can verify the recovered ones.

In [ ]:
def line_through(p1, p2):
    """Homogeneous line through two image points."""
    h1 = torch.cat([p1, torch.ones(1)])
    h2 = torch.cat([p2, torch.ones(1)])
    return torch.linalg.cross(h1, h2)


def intersect(l1, l2):
    """Image-plane intersection of two homogeneous lines (returns inhomogeneous (x, y))."""
    p = torch.linalg.cross(l1, l2)
    return p[:2] / p[2]


def vanishing_point_from_segments(segments_2d):
    """RANSAC-style consensus over pairwise intersections. segments_2d: (N, 2, 2)."""
    lines = [line_through(s[0], s[1]) for s in segments_2d]
    candidates = []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            p = torch.linalg.cross(lines[i], lines[j])
            if abs(p[2]) < 1e-6:
                continue
            candidates.append(p[:2] / p[2])
    candidates = torch.stack(candidates)
    return candidates.median(dim=0).values   # robust to outliers across edge pairs


def ground_truth_vp(D, K, R):
    """Eq. 42.5 — the vanishing point of world direction D."""
    v = K @ R @ torch.as_tensor(D, dtype=torch.float32)
    return v[:2] / v[2]

In [ ]:
# Recover the three orthogonal vanishing points from the projected office edges; compare to ground truth.
# Use only the edges of the room itself — they're the cleanest representatives of the X/Y/Z directions.
room_proj = project_segments(scene["room"], K_true, R_true, eye_true)
edge_dirs_world = scene["room"][:, 1] - scene["room"][:, 0]

# Split edges by which world axis they run along.
axis_indices = {axis: [] for axis in "XYZ"}
for i, d in enumerate(edge_dirs_world):
    axis = "XYZ"[int(torch.argmax(torch.abs(d)))]
    axis_indices[axis].append(i)

recovered = {}
ground_truth = {}
for axis, dir_vec in zip("XYZ", [(1, 0, 0), (0, 1, 0), (0, 0, 1)]):
    segs = room_proj[axis_indices[axis]]
    recovered[axis] = vanishing_point_from_segments(segs)
    ground_truth[axis] = ground_truth_vp(dir_vec, K_true, R_true)

print("axis  |  recovered (px)            |  ground truth (px)")
print("-" * 70)
for axis in "XYZ":
    r = recovered[axis]; g = ground_truth[axis]
    print(f" {axis}     ({r[0]:>10.1f}, {r[1]:>10.1f})  |  ({g[0]:>10.1f}, {g[1]:>10.1f})")

## 42.4 — Measuring heights with the cross-ratio

For any four collinear points the **cross-ratio** is a projective invariant — it survives perspective projection unchanged (equation 42.6):
$$\mathrm{CR}(\mathbf{P}_1, \mathbf{P}_2, \mathbf{P}_3, \mathbf{P}_4) = \frac{|\mathbf{P}_3 - \mathbf{P}_1|\,|\mathbf{P}_4 - \mathbf{P}_2|}{|\mathbf{P}_3 - \mathbf{P}_2|\,|\mathbf{P}_4 - \mathbf{P}_1|}.$$

This is the single trick that powers single-view metrology. We'll demonstrate the invariance, then use it (Algorithm 2 in the chapter) to measure the desk's height from the projected image alone, given only that the bookshelf is 197 cm tall.

In [ ]:
def cross_ratio(p1, p2, p3, p4):
    """Cross-ratio of four collinear points (2D or 1D tensors)."""
    def dist(a, b):
        return torch.linalg.norm(a - b)
    return (dist(p3, p1) * dist(p4, p2)) / (dist(p3, p2) * dist(p4, p1))

### 42.4.2 — Algorithm 2: measuring the desk's height

From the chapter, given image points $\mathbf{g}_b, \mathbf{t}_b$ (bookshelf bottom/top), $\mathbf{g}_d, \mathbf{t}_d$ (desk bottom/top), horizon line $\mathbf{h}$, and vertical vanishing point $\mathbf{v}_3$:

$$\mathbf{l}_1 = \mathbf{g}_d \times \mathbf{g}_b, \quad \mathbf{a} = \mathbf{l}_1 \times \mathbf{h}, \quad \mathbf{l}_2 = \mathbf{a} \times \mathbf{t}_d, \quad \mathbf{l}_3 = \mathbf{g}_b \times \mathbf{t}_b, \quad \mathbf{b} = \mathbf{l}_2 \times \mathbf{l}_3.$$

Then the desk height comes out of the cross-ratio identity:
$$\frac{h_{\text{bookshelf}}}{h_{\text{desk}}} = \frac{|\mathbf{b}-\mathbf{v}_3|\,|\mathbf{t}_b - \mathbf{g}_b|}{|\mathbf{b}-\mathbf{g}_b|\,|\mathbf{t}_b - \mathbf{v}_3|}.$$

In [ ]:
# TODO: pick the correct corner pixels of the projected bookshelf and desk, run Algorithm 2,
#       confirm the recovered desk height comes back close to the ground-truth 76 cm.
#       The pieces (line_through, intersect, cross_ratio, vanishing_point_from_segments)
#       are all in place above. Reproduces Figures 42.13, 42.14, 42.15.
pass

### 42.4.3 — Algorithm 3: height propagation to supported objects

Once we know the desk's height, the bottle (which rests on the desk) becomes measurable too. We project the bottle's height onto the bookshelf via the same construction, but reference the desk top instead of the floor:
$$h_{\text{bottle}} = h_{\text{bookshelf}} \frac{|\mathbf{c}-\mathbf{g}_b|\,|\mathbf{t}_b - \mathbf{v}_3|}{|\mathbf{c}-\mathbf{v}_3|\,|\mathbf{t}_b - \mathbf{g}_b|} - h_{\text{desk}}.$$

Truth: 25 cm. Reproduces figure 42.16.

In [ ]:
# TODO: apply Algorithm 3 to the projected bottle. Should land near 25 cm.
pass

## 42.6 — Camera calibration from three vanishing points

**Algorithm 6.** If the scene contains three mutually orthogonal directions and we have their vanishing points $\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3$, then orthogonality gives three linear constraints on $\mathbf{W} = \mathbf{K}^{-\top}\mathbf{K}^{-1}$ (equation 42.9):
$$\mathbf{v}_i^\top \mathbf{W}\, \mathbf{v}_j = 0 \quad \text{for } (i, j) \in \{(1,2), (1,3), (2,3)\}.$$

Solve for the four free parameters of $\mathbf{W}$ by SVD, then Cholesky-factor $\mathbf{W} = \mathbf{L}^\top\mathbf{L}$ to recover $\mathbf{K} = \mathbf{L}^{-1}$ (normalised so $\mathbf{K}_{33} = 1$).

On synthetic data we can verify the recovered $\mathbf{K}$ matches `K_true` to within a few percent.

In [ ]:
def calibrate_K_from_vanishing_points(v1, v2, v3):
    """Algorithm 6: K from three orthogonal vanishing points (zero skew, square pixels).

    Assumes W = K^-T K^-1 has the form:
        [a, 0, b]
        [0, a, c]
        [b, c, d]
    so w = [a, b, c, d]. Build A from the three orthogonality equations, solve A w = 0 via SVD.
    """
    def homog(v):
        return torch.cat([v, torch.ones(1)])

    def row(vi, vj):
        a, b, _ = homog(vi); c, d, _ = homog(vj)
        # v_i^T W v_j expanded into coefficients of [a_W, b_W, c_W, d_W]
        return torch.tensor([a * c + b * d, a + c, b + d, 1.0])

    A = torch.stack([row(v1, v2), row(v1, v3), row(v2, v3)])
    _, _, Vh = torch.linalg.svd(A)
    w = Vh[-1]                                     # right-singular vector for smallest singular value
    a_W, b_W, c_W, d_W = w
    W = torch.tensor([[a_W, 0.0, b_W], [0.0, a_W, c_W], [b_W, c_W, d_W]])
    W = W / W[2, 2]                                # normalise
    L = torch.linalg.cholesky(W)
    K = torch.linalg.inv(L.T)
    return K / K[2, 2]


# TODO: run calibrate_K_from_vanishing_points on the recovered vanishing points above
#       and compare element-wise against K_true. Reproduces the K matrix the book reports.
#       Note: Cholesky requires W to be positive-definite; synthetic-perfect VPs make it so.

## Open questions before finishing this notebook

Worth resolving with Pantelis on Discord before sinking more time into the figure-specific renders:

1. **Real-photo policy** — am I right to substitute the chapter's office photo with a synthetic 3D render? Pros: ground-truth verifiable; reproducible end-to-end. Cons: doesn't look like the book's exact image. Alternative would be a CC-licensed stock photo with hand-annotated reference points.
2. **Perceptual illusions (Ames room 42.3–42.4, Pisa 42.7)** — leave as descriptive notes, or rebuild as interactive plotly demos? They don't really teach the *algorithm*.
3. **Cross-ratio ruler (42.12)** — verify CR = 1.6 on three projected views of a synthetic ruler? Or replicate the book's exact three viewpoints, knowing we can't match the photograph itself?

## TODOs in this notebook

Tracked here so a next session knows what's left:

- [ ] Section 42.3 — figures 42.5, 42.6, 42.8 (line projection, parallel-line convergence, horizon-line diagrams).
- [ ] Section 42.3.4 — figure 42.9 (cross-product VP method on the synthetic edges).
- [ ] Section 42.4.1 — figures 42.11, 42.12 (cross-ratio diagram + synthetic ruler invariance check).
- [ ] Section 42.4.2 — Algorithm 2 worked example; verify recovered desk height ≈ 76 cm; figures 42.13, 42.14, 42.15.
- [ ] Section 42.4.3 — Algorithm 3 worked example; verify bottle height ≈ 25 cm; figure 42.16.
- [ ] Section 42.5 — axis calibration (Algorithm 4) and 3D point localisation (Algorithm 5); figures 42.17–42.20.
- [ ] Section 42.6 — full calibration round-trip; compare recovered $\mathbf{K}$ against `K_true`.
- [ ] Concluding remarks (42.7).